# 🔢 NumPy — Complete Guide (Beginner → Interview Ready)

**How to use this notebook:** run cells top to bottom. Each section has
1. **Concept** – what and why
2. **Code** – runnable examples
3. **🎯 Interview Points** – things interviewers love to ask

## Table of Contents
1. What is NumPy & why it is fast
2. Creating arrays
3. Array attributes
4. Data types
5. Indexing & slicing
6. View vs Copy
7. Reshaping & dimension manipulation
8. Joining & splitting
9. Math operations & universal functions
10. Broadcasting
11. Aggregations & the `axis` concept
12. Comparison, logical ops, masking
13. Sorting, searching, unique
14. Random numbers
15. Linear algebra
16. Missing values (NaN)
17. Performance & vectorization
18. Memory layout & strides
19. Useful advanced functions
20. File I/O
21. Common pitfalls
22. Interview Q&A (rapid-fire)
23. Practice problems with solutions
24. Cheat sheet

## 1. What is NumPy & why is it fast?

**NumPy** (Numerical Python) is the foundation of Python's scientific stack (Pandas, SciPy, scikit-learn, TensorFlow, PyTorch all build on it).
Its core object is the **`ndarray`** — an N-dimensional, fixed-type, contiguous block of memory.

### Why NumPy is faster than Python lists
| Python list | NumPy array |
|---|---|
| Stores pointers to objects scattered in memory | Stores raw values in one contiguous block |
| Can hold mixed types | Homogeneous (single dtype) |
| Loops run in the Python interpreter | Operations run in compiled C (vectorized) |
| No built-in math on whole collection | Element-wise math, broadcasting, linear algebra |

### 🎯 Interview Points
- `ndarray` = **homogeneous**, **fixed-size**, **contiguous memory**, **strided** view of data.
- Speed comes from: contiguous memory (cache-friendly), C implementation, **vectorization** (no Python loop), SIMD.
- Lists are dynamic and heterogeneous; arrays are not — you can't append in place cheaply (`np.append` creates a new array).

In [1]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.4.4


## 2. Creating Arrays

In [2]:
# From Python lists
a = np.array([1, 2, 3, 4])
b = np.array([[1, 2, 3], [4, 5, 6]])
c = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print("1D:", a)
print("2D:\n", b)
print("3D shape:", c.shape)

1D: [1 2 3 4]
2D:
 [[1 2 3]
 [4 5 6]]
3D shape: (2, 2, 2)


In [3]:
# Built-in constructors
print("zeros:\n", np.zeros((2, 3)))
print("ones:\n", np.ones((2, 2)))
print("full:\n", np.full((2, 3), 7))
print("empty (uninitialised garbage!):\n", np.empty((2, 2)))
print("identity:\n", np.eye(3))          # or np.identity(3)
print("diag:\n", np.diag([1, 2, 3]))

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]
ones:
 [[1. 1.]
 [1. 1.]]
full:
 [[7 7 7]
 [7 7 7]]
empty (uninitialised garbage!):
 [[1. 1.]
 [1. 1.]]
identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
diag:
 [[1 0 0]
 [0 2 0]
 [0 0 3]]


In [4]:
# Ranges and sequences
print("arange :", np.arange(0, 10, 2))            # start, stop(excl), step
print("linspace:", np.linspace(0, 1, 5))          # start, stop(INCL), num points
print("logspace:", np.logspace(0, 3, 4))          # 10^0 .. 10^3

# *_like constructors copy shape & dtype of another array
print("zeros_like:", np.zeros_like(a))
print("full_like :", np.full_like(a, 9))

arange : [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
logspace: [   1.   10.  100. 1000.]
zeros_like: [0 0 0 0]
full_like : [9 9 9 9]


In [5]:
# Random arrays (modern API)
rng = np.random.default_rng(42)
print("uniform [0,1):\n", rng.random((2, 3)))
print("ints        :", rng.integers(0, 10, size=5))
print("normal      :", rng.normal(loc=0, scale=1, size=3))

uniform [0,1):
 [[0.77395605 0.43887844 0.85859792]
 [0.69736803 0.09417735 0.97562235]]
ints        : [7 7 7 7 5]
normal      : [-0.85304393  0.87939797  0.77779194]


### 🎯 Interview Points
- **`arange` vs `linspace`**: `arange` takes a *step* (stop excluded; float steps can give surprising length due to rounding). `linspace` takes *number of points* (stop **included** by default) → prefer it for floats.
- **`np.empty`** doesn't initialise memory — contents are arbitrary. Fast, but dangerous if you forget to fill.
- `np.zeros(3)` → shape `(3,)`; `np.zeros((3,))` same; `np.zeros(3, 3)` is an **error** (second arg is dtype) — need a tuple `(3,3)`.
- `np.array` **copies** by default; `np.asarray` does **not** copy if input is already an ndarray.

## 3. Array Attributes

In [6]:
x = np.arange(12, dtype=np.int32).reshape(3, 4)
print(x)
print("shape   :", x.shape)      # tuple of dimension sizes
print("ndim    :", x.ndim)       # number of axes
print("size    :", x.size)       # total elements
print("dtype   :", x.dtype)
print("itemsize:", x.itemsize, "bytes per element")
print("nbytes  :", x.nbytes, "total bytes")
print("strides :", x.strides, "(bytes to step along each axis)")
print("T       :\n", x.T)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape   : (3, 4)
ndim    : 2
size    : 12
dtype   : int32
itemsize: 4 bytes per element
nbytes  : 48 total bytes
strides : (16, 4) (bytes to step along each axis)
T       :
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]


### 🎯 Interview Points
- `shape` is a tuple; `len(arr)` returns size of **first axis** only.
- `size == np.prod(shape)`; `nbytes == size * itemsize`.
- `arr.shape = (..)` can reshape in place but `reshape()` is preferred.

## 4. Data Types (`dtype`)

Common dtypes: `int8/16/32/64`, `uint8...`, `float16/32/64`, `complex64/128`, `bool`, `str_`, `object`, `datetime64`.
Default: ints → `int64` (platform dependent, `int32` on old Windows), floats → `float64`.

In [7]:
i = np.array([1, 2, 3])
f = np.array([1.5, 2.5, 3.5])
print(i.dtype, f.dtype)

# Type casting
print(f.astype(np.int32))                  # truncates toward zero!
print(np.array([1, 2, 3], dtype=np.float32))

# Upcasting when mixing
print((i + f).dtype)                       # int64 + float64 -> float64
print(np.array([1, "a", 2.5]))             # everything becomes strings!

# Integer overflow (silent!)
u = np.array([250], dtype=np.uint8)
print("uint8 250 + 10 =", u + np.uint8(10))    # wraps around

# Memory savings
big64 = np.zeros(1_000_000, dtype=np.float64)
big32 = np.zeros(1_000_000, dtype=np.float32)
print(big64.nbytes / 1e6, "MB vs", big32.nbytes / 1e6, "MB")

int64 float64
[1 2 3]
[1. 2. 3.]
float64
['1' 'a' '2.5']
uint8 250 + 10 = [4]
8.0 MB vs 4.0 MB


### 🎯 Interview Points
- `astype()` **always returns a copy**.
- Integer overflow in NumPy is **silent** (wrap-around), unlike Python ints which are unlimited.
- Mixed-type arrays are **upcast** to the most general type (bool → int → float → complex → str/object).
- Use smaller dtypes (`float32`, `int8`, `uint8`) to save memory, e.g. images use `uint8`.
- `float` in `dtype=float` means `float64`.

## 5. Indexing & Slicing

In [8]:
arr = np.array([10, 20, 30, 40, 50, 60])
print(arr[0], arr[-1])          # single element
print(arr[1:4])                 # slice [start:stop)
print(arr[::2])                 # every 2nd
print(arr[::-1])                # reversed

10 60
[20 30 40]
[10 30 50]
[60 50 40 30 20 10]


In [9]:
m = np.arange(1, 17).reshape(4, 4)
print(m)
print("m[1, 2]   :", m[1, 2])          # row 1, col 2  (preferred over m[1][2])
print("row 0     :", m[0])
print("col 1     :", m[:, 1])
print("sub-block :\n", m[1:3, 1:3])
print("every other row/col:\n", m[::2, ::2])

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]
m[1, 2]   : 7
row 0     : [1 2 3 4]
col 1     : [ 2  6 10 14]
sub-block :
 [[ 6  7]
 [10 11]]
every other row/col:
 [[ 1  3]
 [ 9 11]]


### Boolean (mask) indexing & Fancy (integer-array) indexing

In [10]:
data = np.array([5, 12, 7, 20, 3, 18])

# Boolean mask
mask = data > 10
print("mask     :", mask)
print("filtered :", data[mask])
print("combined :", data[(data > 5) & (data < 20)])   # use & | ~, NOT and/or, and use parentheses

# Fancy indexing: pass a list/array of indices
print("fancy    :", data[[0, 2, 5]])
print("2D fancy :", m[[0, 1, 2], [1, 2, 3]])          # picks (0,1), (1,2), (2,3)

# Assign using mask
d2 = data.copy()
d2[d2 > 10] = 0
print("assigned :", d2)

# np.where: conditional select
print(np.where(data > 10, "big", "small"))
print("indices where >10:", np.where(data > 10))       # returns tuple of index arrays

mask     : [False  True False  True False  True]
filtered : [12 20 18]
combined : [12  7 18]
fancy    : [ 5  7 18]
2D fancy : [ 2  7 12]
assigned : [5 0 7 0 3 0]
['small' 'big' 'small' 'big' 'small' 'big']
indices where >10: (array([1, 3, 5]),)


### 🎯 Interview Points
- **Basic slicing returns a VIEW** (no copy). **Boolean & fancy indexing return a COPY.**
- Use `&`, `|`, `~` for masks — `and/or/not` raise *"truth value of an array is ambiguous"*.
- `m[1][2]` works but creates an intermediate array; `m[1, 2]` is faster and clearer.
- `np.where(cond)` → indices; `np.where(cond, x, y)` → element-wise select.
- Fancy indexing with 2 index arrays pairs them element-wise (not a cross product). For a cross product use `m[np.ix_(rows, cols)]`.

## 6. View vs Copy ⭐ (Very common interview topic)

In [11]:
orig = np.arange(6)
view = orig[1:4]          # slicing -> view
view[0] = 999
print("orig after modifying view:", orig)      # changed!

cp = orig[1:4].copy()     # explicit copy
cp[0] = -1
print("orig after modifying copy:", orig)      # unchanged

# How to check?
print("view.base is orig :", view.base is orig)
print("np.shares_memory  :", np.shares_memory(orig, view), np.shares_memory(orig, cp))

# Boolean indexing gives a copy
sel = orig[orig > 5]
sel[:] = 0
print("orig after boolean-index edit:", orig)

orig after modifying view: [  0 999   2   3   4   5]
orig after modifying copy: [  0 999   2   3   4   5]
view.base is orig : True
np.shares_memory  : True False
orig after boolean-index edit: [  0 999   2   3   4   5]


### 🎯 Interview Points
| Operation | View / Copy |
|---|---|
| Basic slice `a[1:4]` | **View** |
| `reshape`, `T`, `ravel` (usually), `transpose`, `squeeze` | **View** (when possible) |
| Boolean / fancy indexing | **Copy** |
| `flatten()`, `astype()`, `copy()`, `np.array(a)` | **Copy** |
| Assignment `b = a` | **Same object** (not even a new view!) |

- Check with `arr.base` (None ⇒ owns its data) or `np.shares_memory(a, b)`.
- Views are why NumPy is memory-efficient, but also a source of bugs → call `.copy()` when you need independence.

## 7. Reshaping & Dimension Manipulation

In [12]:
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(2, -1).shape)        # -1 = infer this dimension
print(a.reshape(-1, 6).shape)

m = a.reshape(3, 4)
print("transpose:\n", m.T)
print("ravel  (view if possible):", m.ravel())
print("flatten (always copy)    :", m.flatten())

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 6)
(2, 6)
transpose:
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
ravel  (view if possible): [ 0  1  2  3  4  5  6  7  8  9 10 11]
flatten (always copy)    : [ 0  1  2  3  4  5  6  7  8  9 10 11]


In [13]:
v = np.array([1, 2, 3])
print("original       :", v.shape)
print("newaxis (col)  :", v[:, np.newaxis].shape)      # (3,1)
print("newaxis (row)  :", v[np.newaxis, :].shape)      # (1,3)
print("expand_dims    :", np.expand_dims(v, axis=0).shape)

s = np.zeros((1, 3, 1))
print("squeeze        :", np.squeeze(s).shape)          # removes size-1 axes

t = np.arange(24).reshape(2, 3, 4)
print("swapaxes       :", np.swapaxes(t, 0, 2).shape)
print("transpose axes :", np.transpose(t, (1, 0, 2)).shape)
print("moveaxis       :", np.moveaxis(t, 0, -1).shape)

original       : (3,)
newaxis (col)  : (3, 1)
newaxis (row)  : (1, 3)
expand_dims    : (1, 3)
squeeze        : (3,)
swapaxes       : (4, 3, 2)
transpose axes : (3, 2, 4)
moveaxis       : (3, 4, 2)


### 🎯 Interview Points
- Reshape requires **total size unchanged**; only **one** `-1` allowed.
- `ravel()` → view when possible, `flatten()` → always a copy.
- `reshape` returns a view when memory layout allows, otherwise a copy.
- `a[:, np.newaxis]` (or `None`) turns a 1D array into a column — essential for broadcasting.
- `.T` on a 1D array does **nothing** (still `(n,)`).

## 8. Joining & Splitting

In [14]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("concatenate axis=0:\n", np.concatenate([a, b], axis=0))
print("concatenate axis=1:\n", np.concatenate([a, b], axis=1))
print("vstack:\n", np.vstack([a, b]))
print("hstack:\n", np.hstack([a, b]))
print("stack (new axis) shape:", np.stack([a, b], axis=0).shape)   # (2,2,2)
print("column_stack:\n", np.column_stack([[1, 2, 3], [4, 5, 6]]))

concatenate axis=0:
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]
concatenate axis=1:
 [[1 2 5 6]
 [3 4 7 8]]
vstack:
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]
hstack:
 [[1 2 5 6]
 [3 4 7 8]]
stack (new axis) shape: (2, 2, 2)
column_stack:
 [[1 4]
 [2 5]
 [3 6]]


In [15]:
x = np.arange(12).reshape(3, 4)
print(np.split(x, 3, axis=0))           # 3 equal parts
print(np.split(x, [1, 3], axis=1))      # split at column indices 1 and 3
print(np.array_split(np.arange(10), 3)) # allows unequal parts

[array([[0, 1, 2, 3]]), array([[4, 5, 6, 7]]), array([[ 8,  9, 10, 11]])]
[array([[0],
       [4],
       [8]]), array([[ 1,  2],
       [ 5,  6],
       [ 9, 10]]), array([[ 3],
       [ 7],
       [11]])]
[array([0, 1, 2, 3]), array([4, 5, 6]), array([7, 8, 9])]


### 🎯 Interview Points
- **`concatenate`** joins along an **existing** axis; **`stack`** creates a **new** axis.
- Arrays must match in all dimensions except the joining axis.
- `np.split` needs equal division; `np.array_split` does not.
- `np.append(a, x)` allocates a *new* array every call → don't use in loops; build a list and convert once.

## 9. Math Operations & Universal Functions (ufuncs)

A **ufunc** is a vectorized function operating element-by-element (`np.add`, `np.sqrt`, `np.exp`, ...).

In [16]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b :", a + b)
print("a * b :", a * b)              # element-wise, NOT matrix multiplication
print("b / a :", b / a)
print("b // a:", b // a)
print("a ** 2:", a ** 2)
print("b % 7 :", b % 7)
print("scalar:", a * 10)

a + b : [11 22 33 44]
a * b : [ 10  40  90 160]
b / a : [10. 10. 10. 10.]
b // a: [10 10 10 10]
a ** 2: [ 1  4  9 16]
b % 7 : [3 6 2 5]
scalar: [10 20 30 40]


In [17]:
x = np.array([0, np.pi/2, np.pi])
print("sin :", np.sin(x).round(3))
print("exp :", np.exp([0, 1, 2]))
print("log :", np.log([1, np.e, 10]))
print("sqrt:", np.sqrt([1, 4, 9]))
print("abs :", np.abs([-3, 2, -1]))
print("round/floor/ceil:", np.round(2.567, 1), np.floor(2.7), np.ceil(2.2))
print("clip:", np.clip([1, 5, 10, 15], 3, 12))

# In-place ops and out parameter (saves memory)
z = np.arange(5, dtype=float)
np.multiply(z, 2, out=z)
z += 1
print(z)

sin : [0. 1. 0.]
exp : [1.         2.71828183 7.3890561 ]
log : [0.         1.         2.30258509]
sqrt: [1. 2. 3.]
abs : [3 2 1]
round/floor/ceil: 2.6 2.0 3.0
clip: [ 3  5 10 12]
[1. 3. 5. 7. 9.]


### 🎯 Interview Points
- `*` is **element-wise**. Matrix multiply → `@`, `np.matmul`, or `np.dot`.
- Division `/` always returns float; `//` floor division.
- ufuncs have extra methods: `np.add.reduce(a)`, `np.add.accumulate(a)`, `np.multiply.outer(a, b)`.
- Use `out=` or in-place operators (`+=`) to avoid temporary arrays for large data.
- `np.sqrt(-1)` → `nan` with a warning (not error); `1/0` on arrays → `inf` with warning.

## 10. Broadcasting ⭐⭐ (Most asked NumPy interview topic)

Broadcasting lets NumPy operate on arrays of **different shapes** without copying data.

### The 2 rules (compare shapes from the **rightmost** dimension to the left)
Two dimensions are compatible if:
1. They are **equal**, or
2. One of them is **1** (that array is "stretched").

Missing leading dimensions are treated as 1. If neither condition holds → `ValueError`.

In [18]:
# scalar + array
print(np.array([1, 2, 3]) + 10)

# (3,) + (2,3)  -> (2,3)
row = np.array([1, 2, 3])
mat = np.array([[10, 20, 30], [40, 50, 60]])
print(mat + row)

# (3,1) + (1,4) -> (3,4)  (outer-sum style)
col = np.array([[1], [2], [3]])
r = np.array([[10, 20, 30, 40]])
print(col + r)
print("shape:", (col + r).shape)

[11 12 13]
[[11 22 33]
 [41 52 63]]
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]
shape: (3, 4)


In [19]:
# Incompatible shapes
try:
    np.ones((3, 2)) + np.ones((3,))      # (3,2) vs (3,) -> 2 != 3
except ValueError as e:
    print("Error:", e)

# Fix: give the second array a trailing axis
print((np.ones((3, 2)) + np.ones((3,))[:, None]).shape)

Error: operands could not be broadcast together with shapes (3,2) (3,) 
(3, 2)


In [20]:
# Practical example: normalise each column (z-score) without loops
rng = np.random.default_rng(0)
X = rng.normal(50, 10, size=(5, 3))
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)   # (5,3) - (3,) broadcast
print(X_norm.mean(axis=0).round(10), X_norm.std(axis=0).round(10))

# Pairwise distance matrix via broadcasting
pts = np.array([[0, 0], [3, 4], [6, 8]])
diff = pts[:, None, :] - pts[None, :, :]          # (3,1,2)-(1,3,2) -> (3,3,2)
dist = np.sqrt((diff ** 2).sum(axis=-1))
print(dist)

[-0.  0.  0.] [1. 1. 1.]
[[ 0.  5. 10.]
 [ 5.  0.  5.]
 [10.  5.  0.]]


### 🎯 Interview Points
- Trick: **write shapes right-aligned** and check each column.

```
A      (8, 1, 6, 1)
B         (7, 1, 5)
Result (8, 7, 6, 5)
```
- No data is copied — the stretched axis uses **stride 0**.
- Classic questions: *"Add a vector to each row / column of a matrix"*, *"Compute pairwise distances without loops"*, *"Normalise columns"*.
- Common bug: `(n,)` vs `(n,1)` — `(n,) + (n,1)` gives an `(n,n)` matrix, not element-wise!

## 11. Aggregations & the `axis` Concept ⭐

In [21]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print("sum all     :", m.sum())
print("sum axis=0  :", m.sum(axis=0), " (collapse rows -> per column)")
print("sum axis=1  :", m.sum(axis=1), " (collapse cols -> per row)")
print("keepdims    :", m.sum(axis=1, keepdims=True).shape)

print("mean:", m.mean(), " median:", np.median(m), " std:", m.std().round(3), " var:", m.var().round(3))
print("min/max:", m.min(), m.max(), " ptp(range):", np.ptp(m))
print("prod:", m.prod(), " cumsum:", m.cumsum(), " cumprod axis1:\n", m.cumprod(axis=1))
print("argmin/argmax:", m.argmin(), m.argmax(), " argmax per row:", m.argmax(axis=1))
print("percentile 90:", np.percentile(m, 90), " quantile 0.5:", np.quantile(m, 0.5))

sum all     : 21
sum axis=0  : [5 7 9]  (collapse rows -> per column)
sum axis=1  : [ 6 15]  (collapse cols -> per row)
keepdims    : (2, 1)
mean: 3.5  median: 3.5  std: 1.708  var: 2.917
min/max: 1 6  ptp(range): 5
prod: 720  cumsum: [ 1  3  6 10 15 21]  cumprod axis1:
 [[  1   2   6]
 [  4  20 120]]
argmin/argmax: 0 5  argmax per row: [2 2]
percentile 90: 5.5  quantile 0.5: 3.5


### 🎯 Interview Points
- **`axis=k` means "collapse/aggregate along axis k"** — that axis disappears from the result shape.
  For shape `(2,3)`: `axis=0` → result `(3,)`, `axis=1` → result `(2,)`.
- `keepdims=True` keeps the axis with length 1 → makes broadcasting back easy (e.g. `m / m.sum(axis=1, keepdims=True)`).
- `np.std` default is **population** std (`ddof=0`); sample std needs `ddof=1`. (Pandas defaults to `ddof=1`!)
- `argmax` on a multi-dim array without `axis` returns index into the **flattened** array; use `np.unravel_index` to convert.
- `np.median` is robust to outliers; `mean` is not.

In [22]:
idx = np.argmax(m)
print(idx, np.unravel_index(idx, m.shape))
print("row-normalised:\n", m / m.sum(axis=1, keepdims=True))

5 (np.int64(1), np.int64(2))
row-normalised:
 [[0.16666667 0.33333333 0.5       ]
 [0.26666667 0.33333333 0.4       ]]


## 12. Comparison, Logical Operations & Masking

In [23]:
a = np.array([1, 5, 8, 3, 9])
print(a > 4)
print("any >8:", (a > 8).any(), " all >0:", (a > 0).all())
print("count >4:", (a > 4).sum())              # True == 1
print("logical_and:", np.logical_and(a > 2, a < 9))
print("isin:", np.isin(a, [3, 9, 100]))

# allclose: the right way to compare floats
x = 0.1 + 0.2
print(x == 0.3, np.isclose(x, 0.3), np.allclose([x], [0.3]))

# array_equal: whole-array equality
print(np.array_equal([1, 2, 3], [1, 2, 3]))

[False  True  True False  True]
any >8: True  all >0: True
count >4: 3
logical_and: [False  True  True  True False]
isin: [False False False  True  True]
False True True
True


### 🎯 Interview Points
- `a == b` returns an **array**; use `np.array_equal` / `(a == b).all()` for a single bool.
- Never compare floats with `==` → use `np.isclose` / `np.allclose` (tolerances `rtol=1e-5`, `atol=1e-8`).
- `np.nan == np.nan` is `False` → use `np.isnan`.
- Counting matches: `(cond).sum()`; percentage: `(cond).mean()`.

## 13. Sorting, Searching & Unique

In [24]:
a = np.array([30, 10, 50, 20, 40])
print("sort      :", np.sort(a))
print("argsort   :", np.argsort(a))               # indices that would sort
print("desc sort :", np.sort(a)[::-1])
a_copy = a.copy(); a_copy.sort()                   # in-place
print("top 2     :", a[np.argsort(a)[-2:]])

m = np.array([[3, 1, 2], [9, 7, 8]])
print("sort rows :\n", np.sort(m, axis=1))

# Sort rows of a table by a column
table = np.array([[1, 90], [2, 70], [3, 80]])
print(table[table[:, 1].argsort()])

# Unique
u = np.array([3, 1, 2, 3, 1, 3])
vals, counts = np.unique(u, return_counts=True)
print(vals, counts)
print("unique idx/inverse:", np.unique(u, return_index=True, return_inverse=True))

# searchsorted (binary search on sorted array)
s = np.array([10, 20, 30, 40])
print("searchsorted 25 ->", np.searchsorted(s, 25))

# Set operations
print(np.intersect1d([1,2,3,4],[3,4,5]), np.union1d([1,2],[2,3]), np.setdiff1d([1,2,3],[2]))

sort      : [10 20 30 40 50]
argsort   : [1 3 0 4 2]
desc sort : [50 40 30 20 10]
top 2     : [40 50]
sort rows :
 [[1 2 3]
 [7 8 9]]
[[ 2 70]
 [ 3 80]
 [ 1 90]]
[1 2 3] [2 1 3]
unique idx/inverse: (array([1, 2, 3]), array([1, 2, 0]), array([2, 0, 1, 2, 0, 2]))
searchsorted 25 -> 2
[3 4] [1 2 3] [1 3]


### 🎯 Interview Points
- `np.sort` returns a copy; `arr.sort()` sorts **in place**.
- `argsort` is key for "sort one array by another" and "top-k" problems.
- Default sort is **quicksort (introsort)**, O(n log n), not stable; use `kind='stable'` (mergesort/timsort) when order of ties matters.
- For top-k on huge arrays use `np.partition` / `np.argpartition` → O(n).
- `np.unique` returns **sorted** unique values.
- No descending flag → use `[::-1]` or sort `-a`.

## 14. Random Numbers

In [25]:
# Modern way (recommended since NumPy 1.17)
rng = np.random.default_rng(seed=123)
print(rng.random(3))
print(rng.integers(1, 7, size=5))            # dice rolls (high exclusive)
print(rng.choice([10, 20, 30], size=4, replace=True))
print(rng.permutation(5))
arr = np.arange(5); rng.shuffle(arr); print(arr)
print(rng.normal(0, 1, 3), rng.uniform(5, 10, 2), rng.binomial(10, 0.5, 3))

# Legacy API (still very common in older code / interviews)
np.random.seed(0)
print(np.random.rand(3), np.random.randint(0, 10, 3), np.random.randn(2))

[0.68235186 0.05382102 0.22035987]
[2 2 3 2 3]
[30 20 30 20]
[4 2 3 0 1]
[3 2 0 1 4]
[ 1.1921661  -0.67108968  1.00026942] [8.14970102 9.63703629] [4 6 5]
[0.5488135  0.71518937 0.60276338] [3 7 9] [-1.42232584  1.52006949]


### 🎯 Interview Points
- **Seed** → reproducibility. Same seed ⇒ same sequence.
- New `Generator` API (`default_rng`) vs legacy global `np.random.seed` — the new one is preferred (better algorithm PCG64, no global state).
- `randint(low, high)` → high **exclusive**. `rand` uniform [0,1), `randn` standard normal.
- `shuffle` works in place; `permutation` returns a shuffled copy.
- Train/test split by hand: `idx = rng.permutation(n); train = idx[:int(.8*n)]`.

## 15. Linear Algebra (`np.linalg`)

In [26]:
A = np.array([[2, 1], [1, 3]])
B = np.array([[1, 0], [4, 2]])
v = np.array([1, 2])

print("A @ B :\n", A @ B)
print("A.dot(v):", A.dot(v))
print("dot 1D  :", np.dot([1,2,3],[4,5,6]), " inner/outer:", np.inner([1,2],[3,4]), "\n", np.outer([1,2],[3,4]))
print("det     :", np.linalg.det(A).round(3))
print("inverse :\n", np.linalg.inv(A).round(3))
print("rank    :", np.linalg.matrix_rank(A), " trace:", np.trace(A))
print("norm    :", np.linalg.norm(v), np.linalg.norm(A))

# Solve Ax = b  (never compute inv(A) @ b !)
b = np.array([5, 10])
x = np.linalg.solve(A, b)
print("solve   :", x, " check:", np.allclose(A @ x, b))

# Eigen & SVD
vals, vecs = np.linalg.eig(A)
print("eigenvalues:", vals.round(3))
U, S, Vt = np.linalg.svd(A)
print("singular values:", S.round(3))

A @ B :
 [[ 6  2]
 [13  6]]
A.dot(v): [4 7]
dot 1D  : 32  inner/outer: 11 
 [[3 4]
 [6 8]]
det     : 5.0
inverse :
 [[ 0.6 -0.2]
 [-0.2  0.4]]
rank    : 2  trace: 5
norm    : 2.23606797749979 3.872983346207417
solve   : [1. 3.]  check: True
eigenvalues: [1.382 3.618]
singular values: [3.618 1.382]


### 🎯 Interview Points
- `*` element-wise vs `@` matrix product. `np.dot` on 2D = matmul; on 1D = inner product. Prefer `@`.
- Solve linear systems with **`np.linalg.solve`** — faster and numerically more stable than `inv(A) @ b`.
- `det == 0` ⇒ singular ⇒ `inv` raises `LinAlgError`.
- Matrix multiply shapes: `(m,n) @ (n,p) → (m,p)`.
- `np.linalg.lstsq` for least squares (linear regression): `w = np.linalg.lstsq(X, y, rcond=None)[0]`.
- `np.einsum` can express most tensor operations compactly (see section 19).

## 16. Missing Values (NaN) & Infinity

In [27]:
d = np.array([1.0, np.nan, 3.0, np.nan, 5.0])
print("isnan      :", np.isnan(d))
print("sum        :", d.sum(), " <- nan poisons the result")
print("nansum     :", np.nansum(d), " nanmean:", np.nanmean(d), " nanmax:", np.nanmax(d))
print("drop nan   :", d[~np.isnan(d)])
print("replace nan:", np.nan_to_num(d, nan=0.0))
print("fill mean  :", np.where(np.isnan(d), np.nanmean(d), d))
print("inf        :", np.array([1.0]) / 0 if False else np.inf, np.isinf(np.inf), np.isfinite(d))
print("nan == nan :", np.nan == np.nan, "| type:", type(np.nan))

isnan      : [False  True False  True False]
sum        : nan  <- nan poisons the result
nansum     : 9.0  nanmean: 3.0  nanmax: 5.0
drop nan   : [1. 3. 5.]
replace nan: [1. 0. 3. 0. 5.]
fill mean  : [1. 3. 3. 3. 5.]
inf        : inf True [ True False  True False  True]
nan == nan : False | type: <class 'float'>


### 🎯 Interview Points
- `nan` is a **float** → integer arrays cannot hold NaN.
- `nan != nan`; use `np.isnan`.
- Any arithmetic with NaN gives NaN → use `nan*` functions (`nansum`, `nanmean`, `nanstd`...).
- **Masked arrays** (`np.ma`) are another way to treat missing data.

## 17. Performance & Vectorization ⭐

In [28]:
import time

n = 2_000_000
lst = list(range(n))
arr = np.arange(n)

t = time.perf_counter(); r1 = [x * 2 for x in lst];   t_list = time.perf_counter() - t
t = time.perf_counter(); r2 = arr * 2;                t_np   = time.perf_counter() - t
print(f"list comprehension: {t_list*1000:.1f} ms")
print(f"numpy vectorized  : {t_np*1000:.1f} ms  ->  ~{t_list/t_np:.0f}x faster")

# Loop vs vectorised sum of squares
t = time.perf_counter()
s = 0
for x in arr[:200_000]:
    s += x * x
t_loop = time.perf_counter() - t
t = time.perf_counter(); s2 = (arr[:200_000] ** 2).sum(); t_vec = time.perf_counter() - t
print(f"python loop over ndarray: {t_loop*1000:.1f} ms | vectorised: {t_vec*1000:.2f} ms")

list comprehension: 93.9 ms
numpy vectorized  : 18.2 ms  ->  ~5x faster


python loop over ndarray: 29.0 ms | vectorised: 1.46 ms


### Vectorization tips
1. **Replace Python loops with array operations / ufuncs.**
2. Use **boolean masks** instead of `if` inside loops; `np.where` instead of if/else.
3. Use **broadcasting** instead of nested loops.
4. Pre-allocate (`np.empty`) instead of appending in loops.
5. Avoid iterating over a NumPy array element by element — it's *slower than a list*.
6. Use `out=` / in-place ops for big arrays; use smaller dtypes if precision allows.
7. If truly un-vectorisable → `numba`, Cython, or `np.vectorize` (which is only a convenience wrapper — **it is still a Python loop**, not faster!).

### 🎯 Interview Points
- *"Why is NumPy fast?"* → contiguous memory + compiled C + SIMD + no per-element Python overhead + no type checks per element.
- `np.vectorize` ≠ true vectorization.
- Memory-order: row-major (C-order) default. Iterating along the last axis is cache-friendly.

## 18. Memory Layout & Strides (Advanced)

In [29]:
a = np.arange(12, dtype=np.int64).reshape(3, 4)          # C-order (row-major)
f = np.asfortranarray(a)                                 # F-order (column-major)
print("C strides:", a.strides, " F strides:", f.strides)
print("flags C_CONTIGUOUS:", a.flags['C_CONTIGUOUS'], f.flags['C_CONTIGUOUS'])

# Transpose is free: it just swaps strides
print("a.T strides:", a.T.strides, " shares memory:", np.shares_memory(a, a.T))

# Broadcasting uses stride 0
b = np.broadcast_to(np.array([1, 2, 3]), (4, 3))
print("broadcast_to strides:", b.strides)

# Non-contiguous slice
print("a[:, ::2] contiguous?", a[:, ::2].flags['C_CONTIGUOUS'])
print(np.ascontiguousarray(a[:, ::2]).flags['C_CONTIGUOUS'])

C strides: (32, 8)  F strides: (8, 24)
flags C_CONTIGUOUS: True False
a.T strides: (8, 32)  shares memory: True
broadcast_to strides: (0, 8)
a[:, ::2] contiguous? False
True


### 🎯 Interview Points
- **stride** = number of bytes to jump to reach the next element along an axis.
- C-order (row-major, default) vs Fortran-order (column-major). Summing along the contiguous axis is faster.
- `.T` and `reshape` (when possible) are O(1) — only metadata (shape/strides) changes.
- `ravel()` must copy if array is non-contiguous.

## 19. Useful Advanced Functions

In [30]:
# meshgrid – build coordinate grids
x = np.linspace(-1, 1, 3); y = np.linspace(-1, 1, 3)
X, Y = np.meshgrid(x, y)
print(X, "\n", Y, "\n", (X**2 + Y**2))

# diff, cumsum, gradient
p = np.array([1, 4, 9, 16, 25])
print("diff:", np.diff(p), " gradient:", np.gradient(p))

# histogram & bincount
data = np.array([1, 2, 2, 3, 3, 3, 4, 4, 4, 4])
print("bincount:", np.bincount(data))
print("histogram:", np.histogram(data, bins=4))

# digitize (bucketize)
print("digitize:", np.digitize([0.2, 6.4, 3.0], bins=[0, 1, 5, 10]))

# einsum
A = np.arange(6).reshape(2, 3); B = np.arange(12).reshape(3, 4)
print("einsum matmul equal:", np.allclose(np.einsum('ij,jk->ik', A, B), A @ B))
print("einsum trace/sum   :", np.einsum('ii->', np.eye(3)), np.einsum('ij->', A))

# apply_along_axis, vectorize
print(np.apply_along_axis(lambda r: r.max() - r.min(), 1, A))
f = np.vectorize(lambda v: v ** 2 if v % 2 == 0 else -v)
print(f(np.arange(6)))

# tile / repeat / roll / flip / rot90 / tril / triu / outer / cross
print(np.tile([1, 2], 3), np.repeat([1, 2], 3), np.roll([1, 2, 3, 4], 1), np.flip([1, 2, 3]))
print(np.tril(np.ones((3, 3))), np.cross([1, 0, 0], [0, 1, 0]))

# take, put, choose, select, compress, extract
print(np.take([10, 20, 30], [2, 0]), np.extract([1, 0, 1], [7, 8, 9]))
print(np.select([p < 5, p < 20], ["low", "mid"], default="high"))

# Structured arrays (mini table)
people = np.array([("Ann", 25, 55.5), ("Bob", 30, 80.2)],
                  dtype=[("name", "U10"), ("age", "i4"), ("weight", "f4")])
print(people["name"], people[people["age"] > 26])

[[-1.  0.  1.]
 [-1.  0.  1.]
 [-1.  0.  1.]] 
 [[-1. -1. -1.]
 [ 0.  0.  0.]
 [ 1.  1.  1.]] 
 [[2. 1. 2.]
 [1. 0. 1.]
 [2. 1. 2.]]
diff: [3 5 7 9]  gradient: [3. 4. 6. 8. 9.]
bincount: [0 1 2 3 4]
histogram: (array([1, 2, 3, 4]), array([1.  , 1.75, 2.5 , 3.25, 4.  ]))
digitize: [1 3 2]
einsum matmul equal: True
einsum trace/sum   : 3.0 15
[2 2]
[ 0 -1  4 -3 16 -5]
[1 2 1 2 1 2] [1 1 1 2 2 2] [4 1 2 3] [3 2 1]
[[1. 0. 0.]
 [1. 1. 0.]
 [1. 1. 1.]] [0 0 1]
[30 10] [7 9]
['low' 'low' 'mid' 'mid' 'high']
['Ann' 'Bob'] [('Bob', 30, 80.2)]


### 🎯 Interview Points
- `np.tile` repeats the **whole** array; `np.repeat` repeats **each element**.
- `np.bincount` is very fast for counting non-negative ints.
- `meshgrid` is used for plotting surfaces / evaluating functions on grids (`indexing='xy'` vs `'ij'`).
- `einsum` is a compact & powerful way to do dot products, traces, transposes, batch matmul.

## 20. File I/O

In [31]:
import os, tempfile
tmp = tempfile.mkdtemp()
a = np.arange(6).reshape(2, 3)

np.save(os.path.join(tmp, "a.npy"), a)                       # binary, single array
print(np.load(os.path.join(tmp, "a.npy")))

np.savez(os.path.join(tmp, "multi.npz"), x=a, y=a * 2)       # multiple arrays
z = np.load(os.path.join(tmp, "multi.npz")); print(z["y"])

np.savetxt(os.path.join(tmp, "a.csv"), a, delimiter=",", fmt="%d")   # text
print(np.loadtxt(os.path.join(tmp, "a.csv"), delimiter=",", dtype=int))
# np.genfromtxt(...) handles missing values in CSVs

[[0 1 2]
 [3 4 5]]
[[ 0  2  4]
 [ 6  8 10]]
[[0 1 2]
 [3 4 5]]


### 🎯 Interview Points
- `.npy` binary format is fast & preserves dtype/shape; `.npz` = zip of several arrays.
- `np.loadtxt` fails with missing values → use `np.genfromtxt` (or Pandas).
- For huge files use memory mapping: `np.load(..., mmap_mode='r')` / `np.memmap`.

## 21. Common Pitfalls ⚠️

In [32]:
# 1. Aliasing
a = np.arange(3); b = a; b[0] = 99; print("alias:", a)

# 2. Mutable list-multiplication analogy (lists) vs np.tile
lst = [[0] * 3] * 2; lst[0][0] = 1; print("list shared rows:", lst)

# 3. and/or vs &/|
try:
    if np.array([1, 2]) > 0: pass
except ValueError as e:
    print("ValueError:", e)

# 4. Integer division & dtype
print(np.array([1, 2, 3]) / 2, np.array([1, 2, 3]) // 2)

# 5. Float precision
print(np.arange(0, 1, 0.1).size, np.arange(0, 1.0000001, 0.1).size)

# 6. Shape (n,) vs (n,1)
v = np.array([1, 2, 3])
print((v + v[:, None]).shape, "<- surprise (3,3)")

# 7. Unequal-length list -> object array / error in new NumPy
try:
    np.array([[1, 2], [3]])
except ValueError as e:
    print("ragged:", str(e)[:70], "...")

# 8. np.sum vs python sum on 2D
m = np.ones((2, 2)); print(sum(m), np.sum(m))

alias: [99  1  2]
list shared rows: [[1, 0, 0], [1, 0, 0]]
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
[0.5 1.  1.5] [0 1 1]
10 11
(3, 3) <- surprise (3,3)
ragged: setting an array element with a sequence. The requested array has an i ...
[2. 2.] 4.0


## 22. Interview Q&A (Rapid-fire) 🎤

**Basic**
1. **What is NumPy?** Library for fast N-dimensional array computation with vectorised math, broadcasting, linear algebra, random, FFT.
2. **List vs ndarray?** Homogeneous, contiguous, vectorised, less memory, much faster vs heterogeneous, pointer-based, slow loops.
3. **How to create a 3×3 identity / zeros / random?** `np.eye(3)`, `np.zeros((3,3))`, `np.random.default_rng().random((3,3))`.
4. **`shape` vs `size` vs `ndim`?** Dimensions tuple vs total elements vs number of axes.
5. **`arange` vs `linspace`?** step-based (stop excl.) vs count-based (stop incl.).
6. **`reshape(-1)`?** Flattens (infers length).
7. **`ravel` vs `flatten`?** view (if possible) vs always copy.
8. **`concatenate` vs `stack`?** existing axis vs new axis.

**Intermediate**
9. **Explain broadcasting.** Right-aligned shape comparison; dims equal or 1; zero-copy stretching via stride 0.
10. **View vs copy?** Slices are views; fancy/boolean indexing and `astype`/`flatten`/`copy` are copies.
11. **What does `axis` mean?** The axis that gets collapsed by the operation.
12. **How do you handle NaN?** `np.isnan`, `nan*` functions, `nan_to_num`, boolean mask, `np.ma`.
13. **`np.dot` vs `*` vs `@`?** Matrix/inner product vs element-wise vs matmul operator.
14. **Why is `np.vectorize` not fast?** It's a Python-level loop wrapper.
15. **Deep copy of an array?** `arr.copy()` (for object dtype arrays, `copy.deepcopy`).
16. **How to get indices of the max value in each row?** `arr.argmax(axis=1)`.
17. **How to remove duplicates / count frequency?** `np.unique(arr, return_counts=True)`.
18. **How do you sort a 2D array by a column?** `a[a[:, k].argsort()]`.

**Advanced**
19. **Strides?** Byte steps per axis; transposing/broadcasting only changes strides.
20. **C vs Fortran order?** Row-major vs column-major layout; affects performance & interop with BLAS/LAPACK.
21. **What is a ufunc?** Universal (vectorised) function with `reduce`, `accumulate`, `outer`, `at`, `out=`, broadcasting support.
22. **Memory efficient big data?** Smaller dtypes, `memmap`, chunking, in-place ops, views.
23. **`np.float64` vs Python `float`?** NumPy scalar type (has dtype/methods) vs Python object; both 64-bit IEEE-754 but NumPy scalars slower for single values.
24. **Why `solve` over `inv`?** Faster + more numerically stable.
25. **How to speed up non-vectorizable code?** Numba `@njit`, Cython, rewrite with broadcasting, or use `np.einsum`.
26. **Explain `np.where(cond)` output.** Tuple of index arrays, one per dimension.
27. **What happens with integer overflow?** Silently wraps (fixed-width ints).
28. **Numpy `matrix` vs `ndarray`?** `np.matrix` is deprecated; use 2-D ndarray with `@`.

## 23. Practice Problems (try first, then read the solution!)

In [33]:
# P1. Create a 5x5 matrix with 1 on the border and 0 inside
p1 = np.ones((5, 5)); p1[1:-1, 1:-1] = 0
print(p1)

# P2. Checkerboard 8x8
cb = np.zeros((8, 8), dtype=int); cb[1::2, ::2] = 1; cb[::2, 1::2] = 1
print(cb)

# P3. Replace all odd numbers in an array with -1 (without modifying the original)
a = np.arange(10); out = np.where(a % 2 == 1, -1, a); print(out)

# P4. Normalise a matrix to range [0, 1] (min-max)
m = np.random.default_rng(1).integers(0, 100, (3, 3))
print(((m - m.min()) / (m.max() - m.min())).round(2))

# P5. Find the 2nd largest unique element
x = np.array([5, 1, 9, 9, 3, 7]); print(np.unique(x)[-2])

# P6. Moving average with window 3
s = np.array([1, 2, 3, 4, 5, 6.])
print(np.convolve(s, np.ones(3) / 3, mode="valid"))

# P7. Count of each element per row without loops -> one-hot encoding
labels = np.array([0, 2, 1, 2]); print(np.eye(3, dtype=int)[labels])

# P8. Given 2D array, subtract row mean from each row
z = np.arange(6).reshape(2, 3); print(z - z.mean(axis=1, keepdims=True))

# P9. Indices of the top-3 values
v = np.array([10, 50, 30, 70, 20]); print(np.argsort(v)[-3:][::-1], np.argpartition(v, -3)[-3:])

# P10. Linear regression with NumPy only
rng = np.random.default_rng(0)
Xf = rng.random(50); yf = 3 * Xf + 2 + rng.normal(0, 0.05, 50)
Xd = np.c_[np.ones(50), Xf]
w = np.linalg.lstsq(Xd, yf, rcond=None)[0]
print("intercept, slope:", w.round(2))

# P11. Pairwise Euclidean distance (no loops)
P = rng.random((4, 2))
D = np.sqrt(((P[:, None] - P[None]) ** 2).sum(-1)); print(D.round(2))

# P12. Softmax (numerically stable)
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
print(softmax(np.array([1000., 1001., 1002.])))

[[1. 1. 1. 1. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 1. 1. 1. 1.]]
[[0 1 0 1 0 1 0 1]
 [1 0 1 0 1 0 1 0]
 [0 1 0 1 0 1 0 1]
 [1 0 1 0 1 0 1 0]
 [0 1 0 1 0 1 0 1]
 [1 0 1 0 1 0 1 0]
 [0 1 0 1 0 1 0 1]
 [1 0 1 0 1 0 1 0]]
[ 0 -1  2 -1  4 -1  6 -1  8 -1]
[[0.48 0.52 0.78]
 [1.   0.   0.12]
 [0.86 0.99 0.23]]
7
[2. 3. 4. 5.]
[[1 0 0]
 [0 0 1]
 [0 1 0]
 [0 0 1]]
[[-1.  0.  1.]
 [-1.  0.  1.]]
[3 1 2] [2 1 3]
intercept, slope: [1.99 3.03]
[[0.   0.66 0.36 0.79]
 [0.66 0.   0.43 0.3 ]
 [0.36 0.43 0.   0.45]
 [0.79 0.3  0.45 0.  ]]
[0.09003057 0.24472847 0.66524096]


## 24. Cheat Sheet 📌

| Task | Code |
|---|---|
| Create | `np.array, zeros, ones, full, eye, arange, linspace, random.default_rng().random` |
| Inspect | `.shape .ndim .size .dtype .strides .nbytes` |
| Reshape | `reshape, ravel, flatten, T, newaxis, squeeze, expand_dims` |
| Join / split | `concatenate, stack, vstack, hstack, split, array_split` |
| Index | `a[i, j]`, `a[i:j]`, `a[mask]`, `a[[i, j]]`, `np.where` |
| Math | `+ - * / ** @`, `sqrt exp log sin abs clip round` |
| Aggregate | `sum mean median std var min max argmin argmax cumsum prod percentile` (with `axis`, `keepdims`) |
| Compare | `> < == & \| ~`, `any all isclose allclose isin array_equal` |
| Sort / search | `sort argsort unique searchsorted partition` |
| Linear algebra | `@ dot inv det solve eig svd norm lstsq` |
| NaN | `isnan nansum nanmean nan_to_num` |
| I/O | `save load savez savetxt loadtxt genfromtxt` |

### 🧭 Study Roadmap
1. Master **shapes, indexing, broadcasting, axis** (80% of interview questions).
2. Practice **vectorising loops** and replacing `for` with masks / `where`.
3. Learn **view vs copy** and **dtype** gotchas.
4. Then go to **Pandas → Matplotlib → scikit-learn**.

**Good luck with your interviews! 🚀**